In [ ]:
import pandas
from statsmodels.tsa.stattools import adfuller  
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf


import warnings
warnings.filterwarnings('ignore')  # Suppress all warnings

In [ ]:
df_merged = pd.read_csv("dataset_forecast_revenues_merged.csv")
df_merged["date"] = pd.to_datetime(df_merged["date"], format='ISO8601' )
df_merged = df_merged[df_merged['date']<='2026-01-01'] # I filter only where I have rows without null values

In [ ]:
stationary_cols = []
non_stationary_cols = []

for col in numeric_cols:
    print('---------------------------------------------')
    print(col)
    print('---------------------------------------------')

    test_staticist = adfuller(df_merged[col])[0]
    adf_criticatical_value_5perc = adfuller(df_merged[col])[4]['5%']
    if test_staticist > adf_criticatical_value_5perc:
        non_stationary_cols.append(col)
        print(f'{round(test_staticist,2)} > {round(adf_criticatical_value_5perc ,2)} ==> The time series is NON stationary')
        df_merged[f'{col}_diff'] = df_merged[col] - df_merged[col].shift( 1) # qui facciamo subito la differnziazione

    else:
        stationary_cols.append(col)
        print(f'{round(test_staticist,2)} < {round(adf_criticatical_value_5perc ,2)} ==> The time series is STATIONARY')

In [ ]:
width = 10
height = 5

f, ax = plt.subplots(nrows=2, ncols=1, figsize=(width, 2*height))
plot_acf(df_merged_detrended['TotSales'],lags=36, ax=ax[0], alpha=0.05 )
plot_pacf(df_merged_detrended['TotSales'],lags=36, ax=ax[1], alpha=0.05, method="ywm")

# ax[1].annotate('Strong correlation at lag = 1', xy=(1, 0.6),  xycoords='data',
#             xytext=(0.17, 0.75), textcoords='axes fraction',
#             arrowprops=dict(color='red', shrink=0.05, width=1))

plt.tight_layout()
plt.show()

In [ ]:
df_merged_detrended['rolling_mean_3m'] = df_merged_detrended['TotSales_diff'].rolling(window=3).mean()
df_merged_detrended['rolling_mean_6m'] = df_merged_detrended['TotSales_diff'].rolling(window=6).mean()
df_merged_detrended['rolling_mean_12m'] = df_merged_detrended['TotSales_diff'].rolling(window=12).mean()
df_merged_detrended['rolling_std_12m'] = df_merged_detrended['TotSales_diff'].rolling(window=12).std()

df_merged_detrended['lag_1m'] = df_merged_detrended['TotSales_diff'].shift(1)
df_merged_detrended['lag_2m'] = df_merged_detrended['TotSales_diff'].shift(2)
df_merged_detrended['lag_3m'] = df_merged_detrended['TotSales_diff'].shift(3)
df_merged_detrended['lag_12m'] = df_merged_detrended['TotSales_diff'].shift(12)
